## Test Cell - Upload an image and classify it

In [5]:
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
import torch.nn as nn
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import io, os, json

# ── 1. Load model from checkpoint ─────────────────────────────────────────────
CKPT_PATH = Path(os.path.expanduser(
    '~/Downloads/Capstone_project/model_and_data_pipeline/model_resnet50/primary_checkpoint.pth'
))
print("Log1: Checking the CKPT_PATH:", CKPT_PATH)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print("Log2: The available device is:", DEVICE)

print(f'Loading model from: {CKPT_PATH}')
ckpt          = torch.load(CKPT_PATH, map_location=DEVICE)
IDX_TO_CLASS  = ckpt['idx_to_class']
IMG_SIZE      = ckpt['img_size']
IMAGENET_MEAN = ckpt['imagenet_mean']
IMAGENET_STD  = ckpt['imagenet_std']
NUM_CLASSES   = ckpt['num_classes']

model = resnet50(weights=None)
in_features = model.fc.in_features   # 2048 for ResNet50
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES)
)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE)
model.eval()
print(f'Model loaded. Classes: {list(IDX_TO_CLASS.values())}')

# ── 2. Transform ───────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── 3. Inference function ──────────────────────────────────────────────────────
def predict(image_bytes):
    img    = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred_idx   = int(np.argmax(probs))
    pred_class = IDX_TO_CLASS[pred_idx]
    confidence = float(probs[pred_idx])
    top3       = [(IDX_TO_CLASS[i], float(probs[i]))
                  for i in np.argsort(probs)[::-1][:3]]
    # confidence tier
    is_haz = pred_class == 'hazardous'
    hi     = 0.80 - (0.10 if is_haz else 0)
    med    = 0.55 - (0.10 if is_haz else 0)
    tier   = 'HIGH' if confidence >= hi else ('MEDIUM' if confidence >= med else 'LOW')
    return img, pred_class, confidence, tier, top3

# ── 4. Guidance ────────────────────────────────────────────────────────────────
GUIDANCE = {
    'plastic'      : '♻  Recyclable — clean and place in recycling bin.',
    'paper'        : '♻  Recyclable — flatten and place in paper recycling.',
    'metal'        : '♻  Recyclable — rinse and place in metal recycling bin.',
    'glass'        : '♻  Recyclable — rinse and place in glass recycling bin.',
    'organic'      : '🌱 Organic — compost or dispose in green/organic bin.',
    'e-waste'      : '⚡ E-Waste — take to nearest e-waste collection point.',
    'hazardous'    : '🚨 HAZARDOUS — do NOT mix with regular trash. Take to hazardous waste facility.',
    'general_trash': '🗑  General Trash — dispose in regular waste bin.',
}

# ── 5. Upload widget UI ────────────────────────────────────────────────────────
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp',
    multiple=False,
    description='Upload Image',
    button_style='success'
)
output = widgets.Output()

def on_upload(change):
    with output:
        clear_output(wait=True)
        if not uploader.value:
            return
        file_info   = uploader.value[0]
        image_bytes = file_info['content']
        try:
            img, pred_class, confidence, tier, top3 = predict(image_bytes)
        except Exception as e:
            print(f'Error during prediction: {e}')
            return

        display(img.resize((224, 224)))
        print('=' * 45)
        print(f'  Predicted class  : {pred_class.upper()}')
        print(f'  Confidence       : {confidence*100:.1f}%')
        print(f'  Confidence tier  : {tier}')
        print('=' * 45)
        print('  Top 3:')
        for cls, conf in top3:
            bar = '█' * int(conf * 25)
            print(f'    {cls:<16} {conf*100:5.1f}%  {bar}')
        print('=' * 45)
        if tier == 'LOW':
            print('  ⚠  Low confidence — retake photo with item')
            print('     filling more of the frame.')
        else:
            print(f'  {GUIDANCE.get(pred_class, "")}')

uploader.observe(on_upload, names='value')

print('\nClick the button to upload a waste image:')
display(uploader, output)

Log1: Checking the CKPT_PATH: /Users/prassanna/Downloads/Capstone_project/model_and_data_pipeline/model_resnet50/primary_checkpoint.pth
Log2: The available device is: mps
Loading model from: /Users/prassanna/Downloads/Capstone_project/model_and_data_pipeline/model_resnet50/primary_checkpoint.pth
Model loaded. Classes: ['e-waste', 'general_trash', 'glass', 'hazardous', 'metal', 'organic', 'paper', 'plastic']

Click the button to upload a waste image:


FileUpload(value=(), accept='.jpg,.jpeg,.png,.webp', button_style='success', description='Upload Image')

Output()